# L1 — Knowledge Base Setup

## What You'll Learn

- How to build a text knowledge base for semantic retrieval over policy documents
- How to set up OpenSearch Serverless as a vector store for semantic retrieval
- How to wire a Bedrock Knowledge Base to an S3 data source and sync it

## Overview

AnyCompany's refund agent needs to answer policy questions ("what is the return window for electronics?"). This notebook builds the knowledge layer that powers those answers.

We upload AnyCompany's return-policy documents to S3, then create a Bedrock Knowledge Base backed by an OpenSearch Serverless vector index. The documents are embedded with Titan Text Embeddings V2 so they become semantically searchable. The Knowledge Base ID is stored in SSM Parameter Store so the Refund Agent in L3 can retrieve it at runtime.

## Architecture

```
                        ┌─────────────────────────────────────────┐
                        │           Ingestion (this notebook)      │
                        │                                          │
  S3 Bucket             │                                          │
  └── return-policy/ ───┼──► parse text docs ──► Titan Text        │
                        │                        Embeddings        │
                        │                        V2 (1024d) ───────┼──► OpenSearch Serverless
                        └─────────────────────────────────────────┘    (vector index)
                                                                              │
                        ┌─────────────────────────────────────────┐          │
                        │           Query (L3 Refund Agent)        │          │
                        │                                          │          │
  User question ────────┼──► Bedrock Knowledge Base               │          │
  "What is the          │    (bedrock-agent-runtime.retrieve)  ◄───┼──────────┘
   return policy?"      │         │                               │
                        │         └──► ranked text passages ──────┼──► Refund Agent
                        └─────────────────────────────────────────┘
```

## Steps

1. Upload return-policy documents to S3
2. Create an OpenSearch Serverless vector-search collection
3. Create the IAM role and permissions for the Knowledge Base
4. Create a vector index in OpenSearch
5. Create a Bedrock Knowledge Base with Titan Text Embeddings V2
6. Create a data source and sync it
7. Test retrieval and store the KB ID in SSM Parameter Store

## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

Install required packages.

In [1]:
%pip install --upgrade boto3 opensearch-py==2.7.0 requests-aws4auth==1.3.0 -q

Note: you may need to restart the kernel to use updated packages.


Import libraries and resolve the AWS account ID and region. 

In [2]:
import boto3
import json
import os
import time
from pathlib import Path

sts = boto3.client("sts")
identity = sts.get_caller_identity()
ACCOUNT_ID = identity["Account"]
REGION = boto3.session.Session().region_name or "us-east-1"

print(f"Account: {ACCOUNT_ID}")
print(f"Region:  {REGION}")

Account: 359894365202
Region:  us-west-2


**SageMaker permissions:** The execution role needs EC2 and AOSS VPC endpoint permissions to create the OpenSearch Serverless VPC endpoint. The cell below creates and attaches the required policy if running on SageMaker.

In [3]:
# Grant SageMaker execution role EC2/AOSS VPC endpoint permissions (idempotent)
SAGEMAKER_EXEC_ROLE_NAME = "agent-stack-SageMakerExecutionRole"

vpc_endpoint_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "EC2VpcEndpointAccess",
            "Effect": "Allow",
            "Action": [
                "ec2:DescribeVpcs",
                "ec2:DescribeSubnets",
                "ec2:DescribeSecurityGroups",
                "ec2:DescribeVpcEndpoints",
                "ec2:CreateVpcEndpoint",
                "ec2:DeleteVpcEndpoints",
                "ec2:CreateTags",
            ],
            "Resource": "*",
        },
        {
            "Sid": "AOSSVpcEndpoint",
            "Effect": "Allow",
            "Action": [
                "aoss:CreateVpcEndpoint",
                "aoss:ListVpcEndpoints",
                "aoss:DeleteVpcEndpoint",
                "aoss:BatchGetVpcEndpoint",
            ],
            "Resource": "*",
        },
    ],
}

iam = boto3.client("iam")
VPC_POLICY_NAME = "sagemaker-vpc-endpoint-policy"
VPC_POLICY_ARN = f"arn:aws:iam::{ACCOUNT_ID}:policy/{VPC_POLICY_NAME}"

try:
    iam.create_policy(
        PolicyName=VPC_POLICY_NAME,
        PolicyDocument=json.dumps(vpc_endpoint_policy),
        Description="EC2/AOSS VPC endpoint permissions for SageMaker notebook",
    )
    print(f"Created policy: {VPC_POLICY_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"Policy already exists: {VPC_POLICY_ARN}")

try:
    iam.attach_role_policy(RoleName=SAGEMAKER_EXEC_ROLE_NAME, PolicyArn=VPC_POLICY_ARN)
    print(f"Attached policy to {SAGEMAKER_EXEC_ROLE_NAME}")
except iam.exceptions.NoSuchEntityException:
    print(f"Role {SAGEMAKER_EXEC_ROLE_NAME} not found — skipping (not running on SageMaker)")

# Allow IAM propagation
time.sleep(10)

Policy already exists: arn:aws:iam::359894365202:policy/sagemaker-vpc-endpoint-policy
Attached policy to agent-stack-SageMakerExecutionRole


## Step 1: Create S3 Bucket and Upload Sample Data

Create the S3 bucket that will hold the knowledge base source data. The bucket name is unique per account and region.

In [4]:
BUCKET_NAME = f"composable-agents-kb-{ACCOUNT_ID}-{REGION}"
RETURN_POLICY_PREFIX = "return-policy/"

s3 = boto3.client("s3", region_name=REGION)

# Create S3 bucket
try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    print(f"Created bucket: {BUCKET_NAME}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists: {BUCKET_NAME}")

# Enable encryption on the bucket
s3.put_bucket_encryption(
    Bucket=BUCKET_NAME,
    ServerSideEncryptionConfiguration={
        'Rules': [{
            'ApplyServerSideEncryptionByDefault': {
                'SSEAlgorithm': 'AES256'
            },
            'BucketKeyEnabled': True
        }]
    }
)
print(f"Enabled encryption on bucket: {BUCKET_NAME}")

# Block all public access
s3.put_public_access_block(
    Bucket=BUCKET_NAME,
    PublicAccessBlockConfiguration={
        'BlockPublicAcls': True,
        'IgnorePublicAcls': True,
        'BlockPublicPolicy': True,
        'RestrictPublicBuckets': True,
    }
)
print(f"Blocked public access on bucket: {BUCKET_NAME}")

Bucket already exists: composable-agents-kb-359894365202-us-west-2
Enabled encryption on bucket: composable-agents-kb-359894365202-us-west-2
Blocked public access on bucket: composable-agents-kb-359894365202-us-west-2


Upload AnyCompany's return-policy documents to S3. These will be ingested into the knowledge base.

In [5]:
import mimetypes

def upload_directory(local_dir, s3_prefix):
    """Upload all files from a local directory to S3 under the given prefix."""
    uploaded = []
    for file_path in Path(local_dir).glob("*"):
        if file_path.is_file() and not file_path.name.startswith("."):
            key = f"{s3_prefix}{file_path.name}"
            content_type, _ = mimetypes.guess_type(str(file_path))
            extra_args = {"ContentType": content_type} if content_type else {}
            s3.upload_file(str(file_path), BUCKET_NAME, key, ExtraArgs=extra_args)
            uploaded.append(key)
            print(f"  Uploaded: {key} ({content_type or 'default'})")
    return uploaded

# Remove any stale objects left over from previous runs (e.g. product images
# from an earlier multimodal setup). The text-only KB should ingest ONLY the
# return-policy documents — anything else causes ingestion failures.
print("Cleaning up stale objects outside the return-policy prefix...")
_stale = []
_paginator = s3.get_paginator("list_objects_v2")
for _page in _paginator.paginate(Bucket=BUCKET_NAME):
    for _obj in _page.get("Contents", []):
        if not _obj["Key"].startswith(RETURN_POLICY_PREFIX):
            _stale.append({"Key": _obj["Key"]})
if _stale:
    for _i in range(0, len(_stale), 1000):
        s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": _stale[_i:_i + 1000]})
    print(f"  Removed {len(_stale)} stale object(s).")
else:
    print("  No stale objects found.")

# Upload return policy documents from sample-data folder
print("\nUploading return policy documents...")
policy_files = upload_directory("./sample-data/return-policy", RETURN_POLICY_PREFIX)

print(f"\nTotal files uploaded: {len(policy_files)} docs")

Cleaning up stale objects outside the return-policy prefix...
  No stale objects found.

Uploading return policy documents...
  Uploaded: return-policy/return-refund-policy.txt (text/plain)

Total files uploaded: 1 docs


## Step 2: Create OpenSearch Serverless Collection


Create the OpenSearch Serverless collection with encryption and network policies.

### Prerequisites

This notebook requires a VPC endpoint for OpenSearch Serverless. Access is restricted to VPC only — no public access is permitted.

The cell below will create the VPC endpoint and extract its ID automatically. Update the VPC, subnet, and security group values for your environment:


In [6]:
import subprocess, json as _json

# ──────────────────────────────────────────────────────────────────────────────
# Discover VPC, subnets, and security group programmatically
ec2 = boto3.client("ec2", region_name=REGION)

# Get the default VPC (fall back to first available VPC)
_vpcs = ec2.describe_vpcs(Filters=[{"Name": "isDefault", "Values": ["true"]}])["Vpcs"]
if not _vpcs:
    _vpcs = ec2.describe_vpcs()["Vpcs"]
VPC_ID = _vpcs[0]["VpcId"]

# Get 2 subnets in different AZs
_all_subnets = ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [VPC_ID]}])["Subnets"]
_seen_azs = set()
SUBNET_IDS = []
for _sn in _all_subnets:
    if _sn["AvailabilityZone"] not in _seen_azs:
        SUBNET_IDS.append(_sn["SubnetId"])
        _seen_azs.add(_sn["AvailabilityZone"])
    if len(SUBNET_IDS) >= 2:
        break

# Get the default security group for the VPC
_sgs = ec2.describe_security_groups(Filters=[
    {"Name": "vpc-id", "Values": [VPC_ID]},
    {"Name": "group-name", "Values": ["default"]},
])["SecurityGroups"]
SECURITY_GROUP_IDS = [_sgs[0]["GroupId"]]

VPCE_NAME = "workshop-aoss-vpce"

print(f"VPC:             {VPC_ID}")
print(f"Subnets:         {SUBNET_IDS}")
print(f"Security Group:  {SECURITY_GROUP_IDS}")
# ──────────────────────────────────────────────────────────────────────────────

aoss = boto3.client("opensearchserverless", region_name=REGION)

# Create or retrieve the VPC endpoint
try:
    vpce_response = aoss.create_vpc_endpoint(
        name=VPCE_NAME,
        vpcId=VPC_ID,
        subnetIds=SUBNET_IDS,
        securityGroupIds=SECURITY_GROUP_IDS,
    )
    vpce_id = vpce_response["createVpcEndpointDetail"]["id"]
    print(f"Created VPC endpoint: {vpce_id}")
except aoss.exceptions.ConflictException:
    # Endpoint already exists — look it up
    vpce_list = aoss.list_vpc_endpoints()
    vpce_id = None
    for vpce in vpce_list.get("vpcEndpointSummaries", []):
        if vpce["name"] == VPCE_NAME:
            vpce_id = vpce["id"]
            break
    if not vpce_id:
        raise RuntimeError(f"VPC endpoint '{VPCE_NAME}' exists but could not be found in listing")
    print(f"VPC endpoint already exists: {vpce_id}")

COLLECTION_NAME = "composable-agents-collection"

# Create encryption policy
try:
    aoss.create_security_policy(
        name=f"{COLLECTION_NAME}-enc",
        type="encryption",
        policy=json.dumps({
            "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}],
            "AWSOwnedKey": True,
        }),
    )
    print("Created encryption policy")
except aoss.exceptions.ConflictException:
    print("Encryption policy already exists")

# Create network policy — public access required for notebook index creation
network_policy_body = [{
    "Rules": [
        {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]},
        {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]},
    ],
    "AllowFromPublic": True,
}]

print("Network policy: public access enabled (required for notebook index creation)")

try:
    aoss.create_security_policy(
        name=f"{COLLECTION_NAME}-net",
        type="network",
        policy=json.dumps(network_policy_body),
    )
    print("Created network policy (public access \u2713)")
except aoss.exceptions.ConflictException:
    # Policy already exists — update it; ignore if content is already identical
    existing = aoss.get_security_policy(name=f"{COLLECTION_NAME}-net", type="network")
    try:
        aoss.update_security_policy(
            name=f"{COLLECTION_NAME}-net",
            type="network",
            policyVersion=existing["securityPolicyDetail"]["policyVersion"],
            policy=json.dumps(network_policy_body),
        )
        print("Updated network policy to allow public access \u2713")
    except aoss.exceptions.ValidationException as e:
        if "no changes detected" in str(e).lower():
            print("Network policy already exists with correct settings \u2713")
        else:
            raise

# Create the collection
try:
    collection_response = aoss.create_collection(
        name=COLLECTION_NAME,
        type="VECTORSEARCH",
    )
    print(f"Creating collection: {COLLECTION_NAME}")
except aoss.exceptions.ConflictException:
    print(f"Collection already exists: {COLLECTION_NAME}")

VPC:             vpc-0b8607222ffe591d6
Subnets:         ['subnet-0229f509f7e90e479', 'subnet-01f2c886c196be673']
Security Group:  ['sg-090f6a14b82613de6']
VPC endpoint already exists: vpce-0d3c9b39b5f151cb9
Encryption policy already exists
Network policy: public access enabled (required for notebook index creation)
Updated network policy to allow public access ✓
Collection already exists: composable-agents-collection


Wait for the collection to reach ACTIVE status before proceeding 

**NOTE:** collection creation typically takes 5–6 minutes.

In [7]:
# Wait for collection to become ACTIVE
import time as _time

print("Waiting for collection to become ACTIVE...")
_wait_start = _time.time()
_poll_count = 0

while True:
    collections = aoss.batch_get_collection(names=[COLLECTION_NAME])
    status = collections["collectionDetails"][0]["status"]
    _elapsed = _time.time() - _wait_start
    _poll_count += 1
    if status == "ACTIVE":
        break
    print(f"  [{_elapsed:5.0f}s] Status: {status}... (poll #{_poll_count})")
    time.sleep(15)

_total = _time.time() - _wait_start
collection_detail = collections["collectionDetails"][0]
COLLECTION_ARN = collection_detail["arn"]
COLLECTION_ENDPOINT = collection_detail["collectionEndpoint"]
COLLECTION_ID = collection_detail["id"]

print(f"Collection ACTIVE after {_total:.1f}s ({_poll_count} poll(s))")
print(f"  ARN: {COLLECTION_ARN}")
print(f"  Endpoint: {COLLECTION_ENDPOINT}")

Waiting for collection to become ACTIVE...
Collection ACTIVE after 0.0s (1 poll(s))
  ARN: arn:aws:aoss:us-west-2:359894365202:collection/t5x2c9c2o4p1opriibia
  Endpoint: https://t5x2c9c2o4p1opriibia.us-west-2.aoss.amazonaws.com


## Step 3: Create IAM Role for Bedrock Knowledge Base

Create the IAM role that Bedrock will assume to access S3, OpenSearch Serverless, and Bedrock Data Automation on behalf of the Knowledge Base.

In [8]:
iam = boto3.client("iam")

KB_ROLE_NAME = "composable-agents-kb-role"

# NOTE: Use only aws:SourceAccount in the trust policy condition.
# aws:SourceArn with knowledge-base/* fails on the initial CreateKnowledgeBase
# call because the KB doesn't exist yet and Bedrock can't populate aws:SourceArn,
# which causes: 'Bedrock Knowledge Base was unable to assume the given role.'
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID},
            },
        }
    ],
}

try:
    role_response = iam.create_role(
        RoleName=KB_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Role for Bedrock Knowledge Base with OpenSearch Serverless",
    )
    KB_ROLE_ARN = role_response["Role"]["Arn"]
    print(f"Created role: {KB_ROLE_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    KB_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{KB_ROLE_NAME}"
    # Update the trust policy on the existing role to fix any prior bad condition
    iam.update_assume_role_policy(
        RoleName=KB_ROLE_NAME,
        PolicyDocument=json.dumps(trust_policy),
    )
    print(f"Role already exists, trust policy updated: {KB_ROLE_ARN}")

Role already exists, trust policy updated: arn:aws:iam::359894365202:role/composable-agents-kb-role


Build and attach the IAM policy granting the KB role access to the S3 data bucket, Bedrock models, and OpenSearch.

In [9]:
kb_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "S3DataBucket",
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:ListBucket",
            ],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Sid": "BedrockModels",
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel"],
            "Resource": [
                f"arn:aws:bedrock:{REGION}::foundation-model/*",
            ],
        },
        {
            "Sid": "OpenSearchServerlessAccess",
            "Effect": "Allow",
            "Action": ["aoss:APIAccessAll"],
            "Resource": [COLLECTION_ARN],
        },
    ],
}

POLICY_NAME = "composable-agents-kb-policy"
POLICY_ARN = f"arn:aws:iam::{ACCOUNT_ID}:policy/{POLICY_NAME}"

try:
    policy_response = iam.create_policy(
        PolicyName=POLICY_NAME,
        PolicyDocument=json.dumps(kb_policy),
    )
    POLICY_ARN = policy_response["Policy"]["Arn"]
    print(f"Created policy: {POLICY_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    versions = iam.list_policy_versions(PolicyArn=POLICY_ARN)["Versions"]
    if len(versions) >= 5:
        oldest_non_default = [v for v in versions if not v["IsDefaultVersion"]]
        if oldest_non_default:
            iam.delete_policy_version(PolicyArn=POLICY_ARN, VersionId=oldest_non_default[-1]["VersionId"])
    iam.create_policy_version(
        PolicyArn=POLICY_ARN,
        PolicyDocument=json.dumps(kb_policy),
        SetAsDefault=True,
    )
    print(f"Updated policy: {POLICY_ARN}")

iam.attach_role_policy(RoleName=KB_ROLE_NAME, PolicyArn=POLICY_ARN)
print("Policy attached to role.")

# Allow time for IAM propagation
time.sleep(30)

Updated policy: arn:aws:iam::359894365202:policy/composable-agents-kb-policy
Policy attached to role.


## Step 4: Create Data Access Policy for OpenSearch Serverless

Create the OpenSearch Serverless data access policy granting the KB role and your current IAM identity read/write access to the collection's index.

In [10]:
WORKSHOP_STUDIO_ARN= f"arn:aws:sts::{ACCOUNT_ID}:assumed-role/WSParticipantRole/Participant" #Required for workshop studio
SAGEMAKER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/agent-stack-SageMakerExecutionRole/SageMaker" #Required for SageMaker notebooks
data_access_policy = json.dumps([
    {
        "Rules": [
            {
                "ResourceType": "index",
                "Resource": [f"index/{COLLECTION_NAME}/*"],
                "Permission": [
                    "aoss:CreateIndex",
                    "aoss:DeleteIndex",
                    "aoss:UpdateIndex",
                    "aoss:DescribeIndex",
                    "aoss:ReadDocument",
                    "aoss:WriteDocument",
                ],
            },
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{COLLECTION_NAME}"],
                "Permission": [
                    "aoss:CreateCollectionItems",
                    "aoss:DeleteCollectionItems",
                    "aoss:DescribeCollectionItems",
                    "aoss:UpdateCollectionItems",
                ],
            },
        ],
        "Principal": [
            KB_ROLE_ARN,
            identity["Arn"],
            WORKSHOP_STUDIO_ARN,
            SAGEMAKER_ROLE_ARN,
        ],
    }
])

ACCESS_POLICY_NAME = "comp-agents-kb-access"

try:
    aoss.create_access_policy(
        name=ACCESS_POLICY_NAME,
        type="data",
        policy=data_access_policy,
    )
    print("Created data access policy")
except aoss.exceptions.ConflictException:
    try:
        existing = aoss.get_access_policy(name=ACCESS_POLICY_NAME, type="data")
        aoss.update_access_policy(
            name=ACCESS_POLICY_NAME,
            type="data",
            policyVersion=existing["accessPolicyDetail"]["policyVersion"],
            policy=data_access_policy,
        )
        print("Updated data access policy")
    except aoss.exceptions.ValidationException:
        print("Data access policy already up to date")

Data access policy already up to date


## Step 5: Create Vector Index in OpenSearch

Create the kNN vector index in OpenSearch using FAISS HNSW with 1024 dimensions (matching Titan Text Embeddings V2). Retries with backoff because the data access policy takes a few seconds to propagate.

In [11]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

credentials = boto3.Session().get_credentials().get_frozen_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    REGION,
    "aoss",
    session_token=credentials.token,
)

host = COLLECTION_ENDPOINT.replace("https://", "")

os_client = OpenSearch(
    hosts=[{"host": host, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
)

VECTOR_INDEX_NAME = "bedrock-knowledge-base-default-index"
EMBEDDING_DIMENSION = 1024  # Titan Text Embeddings V2 output dimension

# Delete old index if it exists
try:
    os_client.indices.delete(index=VECTOR_INDEX_NAME)
    print(f"Deleted old index: {VECTOR_INDEX_NAME}")
except Exception:
    pass

index_body = {
    "settings": {
        "index": {"knn": True}
    },
    "mappings": {
        "properties": {
            "bedrock-knowledge-base-default-vector": {
                "type": "knn_vector",
                "dimension": EMBEDDING_DIMENSION,
                "method": {
                    "engine": "faiss",
                    "name": "hnsw",
                    "space_type": "l2",
                },
            },
            "AMAZON_BEDROCK_TEXT": {"type": "text"},
            "AMAZON_BEDROCK_METADATA": {"type": "text"},
        }
    },
}

# Retry with backoff — data access policy takes time to propagate
for attempt in range(5):
    try:
        os_client.indices.create(index=VECTOR_INDEX_NAME, body=index_body)
        print(f"Created vector index: {VECTOR_INDEX_NAME} (dimension={EMBEDDING_DIMENSION})")
        break
    except Exception as e:
        if "already_exists" in str(e).lower() or "resource_already_exists" in str(e).lower():
            print(f"Vector index already exists: {VECTOR_INDEX_NAME}")
            break
        if "authorization_exception" in str(e).lower() and attempt < 4:
            wait = 15 * (attempt + 1)
            print(f"  Access policy still propagating, retrying in {wait}s... (attempt {attempt + 1}/5)")
            time.sleep(wait)
        else:
            raise

# OpenSearch Serverless is eventually consistent — the index can take 30-60s
# after creation before it is visible to other services (like Bedrock).
# Without this wait, Step 6 can fail with: 'index bedrock-knowledge-base-default-index does not exist'.
# Poll both indices.exists and a no-op search to confirm the index is fully queryable.
print("\nVerifying vector index is queryable (OpenSearch Serverless eventual consistency)...")
_verify_start = time.time()
_verify_timeout = 180  # 3 minutes
_verified = False
while time.time() - _verify_start < _verify_timeout:
    try:
        if os_client.indices.exists(index=VECTOR_INDEX_NAME):
            # exists() can return True before the index is fully queryable; do a real call.
            os_client.search(index=VECTOR_INDEX_NAME, body={"query": {"match_all": {}}, "size": 0})
            _verified = True
            break
    except Exception as _e:
        _msg = str(_e).lower()
        if "not_found" not in _msg and "index_not_found" not in _msg and "no such index" not in _msg:
            # Surface unexpected errors instead of looping silently
            print(f"  Unexpected error while verifying index: {_e}")
    elapsed = int(time.time() - _verify_start)
    print(f"  [{elapsed:3d}s] Index not yet queryable, waiting...")
    time.sleep(10)

if not _verified:
    raise RuntimeError(
        f"Vector index '{VECTOR_INDEX_NAME}' did not become queryable within {_verify_timeout}s. "
        "Re-run this cell or check the OpenSearch Serverless collection status."
    )

# Extra cushion before the Bedrock control plane reads the index in Step 6.
time.sleep(30)
print(f"Vector index is queryable after {int(time.time() - _verify_start)}s — safe to create Knowledge Base.")

Deleted old index: bedrock-knowledge-base-default-index
Created vector index: bedrock-knowledge-base-default-index (dimension=1024)

Verifying vector index is queryable (OpenSearch Serverless eventual consistency)...
Vector index is queryable after 30s — safe to create Knowledge Base.


### Security hardening: Disable public access

Now that the vector index is created, we no longer need public access from the notebook. Update the network policy to restrict access to the VPC endpoint only, while explicitly allowing the Bedrock service to reach the collection for ingestion and queries.

In [12]:
# Restrict network policy: disable public access, allow VPC endpoint + Bedrock service
restricted_network_policy = [{
    "Rules": [
        {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]},
        {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]},
    ],
    "AllowFromPublic": False,
    "SourceVPCEs": [vpce_id],
    "SourceServices": ["bedrock.amazonaws.com"],
}]

existing = aoss.get_security_policy(name=f"{COLLECTION_NAME}-net", type="network")
aoss.update_security_policy(
    name=f"{COLLECTION_NAME}-net",
    type="network",
    policyVersion=existing["securityPolicyDetail"]["policyVersion"],
    policy=json.dumps(restricted_network_policy),
)
print("Network policy updated: public access disabled")
print(f"  Access restricted to VPC endpoint ({vpce_id}) and Bedrock service")

Network policy updated: public access disabled
  Access restricted to VPC endpoint (vpce-0d3c9b39b5f151cb9) and Bedrock service


## Step 6: Create Bedrock Knowledge Base

Using `amazon.titan-embed-text-v2:0` for text embeddings with `FLOAT32` data type. Text documents are parsed and embedded directly.

In [13]:
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)

KB_NAME = "composable-agents-kb"
EMBEDDING_MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/amazon.titan-embed-text-v2:0"

# If a KB with this name already exists, delete it (and its data sources)
# and wait for the deletion to fully complete before creating a new one.
def delete_kb_if_exists(kb_name, timeout=600):
    """Delete any existing KB with the given name and block until it is gone."""
    old_kb_id = None
    try:
        paginator = bedrock_agent.get_paginator("list_knowledge_bases")
        for page in paginator.paginate():
            for kb in page["knowledgeBaseSummaries"]:
                if kb["name"] == kb_name:
                    old_kb_id = kb["knowledgeBaseId"]
                    break
            if old_kb_id:
                break
    except Exception as e:
        print(f"Could not list knowledge bases: {e}")
        return

    if not old_kb_id:
        print(f"No existing KB named '{kb_name}' — creating a new one.")
        return

    print(f"Existing KB found: {old_kb_id} — deleting before recreating.")

    # Delete data sources first (children before parent)
    try:
        ds_list = bedrock_agent.list_data_sources(knowledgeBaseId=old_kb_id)
        for ds in ds_list["dataSourceSummaries"]:
            bedrock_agent.delete_data_source(knowledgeBaseId=old_kb_id, dataSourceId=ds["dataSourceId"])
            print(f"  Deleted old data source: {ds['name']}")
    except Exception as e:
        print(f"  Data source cleanup note: {e}")

    # Delete the KB and poll until it is fully gone
    try:
        bedrock_agent.delete_knowledge_base(knowledgeBaseId=old_kb_id)
    except bedrock_agent.exceptions.ResourceNotFoundException:
        print("  KB already gone.")
        return
    except Exception as e:
        print(f"  Delete call note: {e}")

    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            status = bedrock_agent.get_knowledge_base(knowledgeBaseId=old_kb_id)["knowledgeBase"]["status"]
            print(f"  Old KB status: {status}...")
        except bedrock_agent.exceptions.ResourceNotFoundException:
            print(f"  Old KB {old_kb_id} fully deleted.")
            return
        time.sleep(10)
    raise RuntimeError(
        f"Timed out after {timeout}s waiting for old KB '{old_kb_id}' to delete. "
        "Re-run this cell once it finishes deleting."
    )

delete_kb_if_exists(KB_NAME)

kb_config = {
    "type": "VECTOR",
    "vectorKnowledgeBaseConfiguration": {
        "embeddingModelArn": EMBEDDING_MODEL_ARN,
        "embeddingModelConfiguration": {
            "bedrockEmbeddingModelConfiguration": {
                "embeddingDataType": "FLOAT32",
            }
        },
    },
}

storage_config = {
    "type": "OPENSEARCH_SERVERLESS",
    "opensearchServerlessConfiguration": {
        "collectionArn": COLLECTION_ARN,
        "vectorIndexName": VECTOR_INDEX_NAME,
        "fieldMapping": {
            "vectorField": "bedrock-knowledge-base-default-vector",
            "textField": "AMAZON_BEDROCK_TEXT",
            "metadataField": "AMAZON_BEDROCK_METADATA",
        },
    },
}

# Retry — IAM policy / trust policy propagation can take up to 60s,
# and OpenSearch Serverless index visibility to Bedrock can lag a bit longer.
for attempt in range(5):
    try:
        kb_response = bedrock_agent.create_knowledge_base(
            name=KB_NAME,
            roleArn=KB_ROLE_ARN,
            knowledgeBaseConfiguration=kb_config,
            storageConfiguration=storage_config,
        )
        KB_ID = kb_response["knowledgeBase"]["knowledgeBaseId"]
        print(f"Created Knowledge Base: {KB_ID}")
        break
    except bedrock_agent.exceptions.ValidationException as e:
        msg = str(e).lower()
        retryable = (
            "unable to assume" in msg
            or "assume the given role" in msg
            or "not authorized" in msg
            or "does not exist" in msg  # vector index not yet visible to Bedrock
            or "no such index" in msg
            or "index_not_found" in msg
        )
        if retryable and attempt < 4:
            wait = 20 * (attempt + 1)
            print(f"  IAM/trust policy or index visibility still propagating, retrying in {wait}s... (attempt {attempt + 1}/5)")
            time.sleep(wait)
        else:
            raise

No existing KB named 'composable-agents-kb' — creating a new one.
Created Knowledge Base: BAQBCKQYIC


## Step 7: Create Data Source

Create the data source pointing at the S3 bucket, scoped to the `return-policy/` prefix so only text documents are ingested. The default parser handles the text documents, chunking and embedding them for semantic search.

In [14]:
ds_response = bedrock_agent.create_data_source(
    knowledgeBaseId=KB_ID,
    name="composable-agents-data",
    dataDeletionPolicy="RETAIN",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{BUCKET_NAME}",
            "inclusionPrefixes": [RETURN_POLICY_PREFIX],
        },
    },
)
DS_ID = ds_response["dataSource"]["dataSourceId"]
print(f"Created data source: {DS_ID}")

Created data source: WQHQWQK3ON


## Step 8: Sync Data Source (Ingest Documents)

Wait for the KB and data source to be ready, then start the ingestion job to embed and index all documents. 

**NOTE:** This typically takes a few minutes.

In [15]:
def wait_for_kb_ready(kb_id, timeout=300):
    """Wait for Knowledge Base to reach ACTIVE status."""
    print("Checking Knowledge Base status...")
    elapsed = 0
    while elapsed < timeout:
        kb = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)
        status = kb["knowledgeBase"]["status"]
        if status == "ACTIVE":
            print(f"  Knowledge Base {kb_id} is ACTIVE")
            return True
        elif status in ("FAILED", "DELETE_IN_PROGRESS"):
            print(f"  Knowledge Base is in {status} state. Cannot start ingestion.")
            return False
        print(f"  KB status: {status}, waiting...")
        time.sleep(15)
        elapsed += 15
    print(f"  Timed out waiting for KB to become ACTIVE (waited {timeout}s)")
    return False

def wait_for_ds_ready(kb_id, ds_id, timeout=300):
    """Wait for Data Source to reach AVAILABLE status."""
    print("Checking Data Source status...")
    elapsed = 0
    while elapsed < timeout:
        ds = bedrock_agent.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
        status = ds["dataSource"]["status"]
        if status == "AVAILABLE":
            print(f" KB: {kb_id} and Data Source: {ds_id} are AVAILABLE")
            return True
        elif status in ("FAILED", "DELETE_IN_PROGRESS"):
            print(f"  Data Source is in {status} state. Cannot start ingestion.")
            return False
        print(f"  Data Source status: {status}, waiting...")
        time.sleep(15)
        elapsed += 15
    print(f"  Timed out waiting for Data Source to become AVAILABLE (waited {timeout}s)")
    return False

def sync_and_wait(kb_id, ds_id, label):
    """Start ingestion job and wait for completion, reporting elapsed time each poll."""
    import time as _time
    response = bedrock_agent.start_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id
    )
    job_id = response["ingestionJob"]["ingestionJobId"]
    print(f"[{label}] Ingestion started: {job_id}")

    _job_start = _time.time()
    _poll = 0
    while True:
        job = bedrock_agent.get_ingestion_job(
            knowledgeBaseId=kb_id,
            dataSourceId=ds_id,
            ingestionJobId=job_id,
        )
        status = job["ingestionJob"]["status"]
        _elapsed = _time.time() - _job_start
        _poll += 1
        if status in ("COMPLETE", "FAILED", "STOPPED"):
            break
        print(f"  [{_elapsed:5.0f}s] Status: {status}... (poll #{_poll})")
        time.sleep(15)

    _total = _time.time() - _job_start
    stats = job["ingestionJob"].get("statistics", {})
    print(f"[{label}] Final status: {status} — completed in {_total:.1f}s ({_poll} poll(s))")
    print(f"  Documents scanned: {stats.get('numberOfDocumentsScanned', 'N/A')}")
    print(f"  Documents indexed: {stats.get('numberOfNewDocumentsIndexed', 0) + stats.get('numberOfModifiedDocumentsIndexed', 0)}")
    print(f"  Documents failed:  {stats.get('numberOfDocumentsFailed', 0)}")
    return status

# Verify KB and Data Source are ready before starting ingestion
kb_ready = wait_for_kb_ready(KB_ID)
ds_ready = wait_for_ds_ready(KB_ID, DS_ID) if kb_ready else False

if kb_ready and ds_ready:
    print("\nSyncing data source (return policy docs)...")
    import time as _time
    _sync_start = _time.time()
    sync_and_wait(KB_ID, DS_ID, "All Data")
    print(f"\nTotal sync wall-clock time: {_time.time() - _sync_start:.1f}s")
else:
    print("\n❌ Cannot start ingestion — KB or Data Source is not in a ready state.")

Checking Knowledge Base status...
  KB status: CREATING, waiting...
  Knowledge Base BAQBCKQYIC is ACTIVE
Checking Data Source status...
 KB: BAQBCKQYIC and Data Source: WQHQWQK3ON are AVAILABLE

Syncing data source (return policy docs)...
[All Data] Ingestion started: MSRI189WMH
  [    0s] Status: STARTING... (poll #1)
[All Data] Final status: COMPLETE — completed in 15.1s (2 poll(s))
  Documents scanned: 1
  Documents indexed: 1
  Documents failed:  0

Total sync wall-clock time: 16.2s


## Step 9: Verify Ingestion & Test Retrieval

Check the ingestion job results — confirms how many documents were scanned, indexed, and whether any failed.

In [16]:
# Check ingestion job results
jobs = bedrock_agent.list_ingestion_jobs(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
for job in jobs["ingestionJobSummaries"]:
    stats = job.get("statistics", {})
    print(f"Job {job['ingestionJobId']}:")
    print(f"  Status: {job['status']}")
    print(f"  Scanned:  {stats.get('numberOfDocumentsScanned', 0)}")
    print(f"  Indexed:  {stats.get('numberOfNewDocumentsIndexed', 0) + stats.get('numberOfModifiedDocumentsIndexed', 0)}")
    print(f"  Failed:   {stats.get('numberOfDocumentsFailed', 0)}")
    print()

Job MSRI189WMH:
  Status: COMPLETE
  Scanned:  1
  Indexed:  1
  Failed:   0



Run three test queries against the knowledge base to confirm the return policy documents are retrievable.

In [17]:
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)


def query_knowledge_base(kb_id, query, num_results=5):
    """Retrieve relevant chunks from a knowledge base."""
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=kb_id,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {"numberOfResults": num_results}
        },
    )
    return response["retrievalResults"]


def print_result(i, result):
    score = result.get("score", 0)
    content = result.get("content", {})
    source = result.get("location", {}).get("s3Location", {}).get("uri", "unknown")
    print(f"\n--- Result {i} (score: {score:.4f}) ---")
    print(f"  Source: {source}")
    if "text" in content:
        print(f"  Text: {content['text'][:200]}")
    elif "byteContent" in content:
        print(f"  Type: {content.get('type', 'unknown')} (binary content)")
    else:
        print(f"  Content keys: {list(content.keys())}")


# --- Warm-up: OpenSearch Serverless is eventually consistent ---
# Freshly ingested vectors can take a few seconds to become searchable. Poll a
# probe query until results appear (capped at 90s) so the demo queries below do
# not return an empty set on the very first run right after ingestion.
import time
print("Warming up retrieval (waiting for ingested vectors to become searchable)...")
_warm_start = time.time()
while time.time() - _warm_start < 90:
    if query_knowledge_base(KB_ID, "return policy", num_results=1):
        print(f"  Retrieval ready after {time.time() - _warm_start:.0f}s\n")
        break
    time.sleep(5)
else:
    print("  Warm-up timed out; retrieval may be empty on the first attempt.\n")

# Test 1: Text query about return policy
print("=" * 60)
print("Query 1: 'What is the return policy for electronics?'")
print("=" * 60)
results = query_knowledge_base(KB_ID, "What is the return policy for electronics?")
for i, result in enumerate(results, 1):
    print_result(i, result)

# Test 2: Return window query
print("\n" + "=" * 60)
print("Query 2: 'How many days do I have to return an item?'")
print("=" * 60)
results = query_knowledge_base(KB_ID, "How many days do I have to return an item?")
for i, result in enumerate(results, 1):
    print_result(i, result)

# Test 3: Refund process query
print("\n" + "=" * 60)
print("Query 3: 'How are refunds processed?'")
print("=" * 60)
results = query_knowledge_base(KB_ID, "How are refunds processed?")
for i, result in enumerate(results, 1):
    print_result(i, result)

Warming up retrieval (waiting for ingested vectors to become searchable)...
  Retrieval ready after 6s

Query 1: 'What is the return policy for electronics?'

--- Result 1 (score: 0.4402) ---
  Source: s3://composable-agents-kb-359894365202-us-west-2/return-policy/return-refund-policy.txt
  Text: RETURN POLICY  Membership Tier Return Windows: - Gold Tier: 30-day return window from delivery date - Silver Tier: 20-day return window from delivery date - Bronze Tier: 15-day return window from deli

--- Result 2 (score: 0.4394) ---
  Source: s3://composable-agents-kb-359894365202-us-west-2/return-policy/return-refund-policy.txt
  Text: Include all parts, accessories, and documentation 7. Drop off at any authorized shipping location  Refund Timeline: - Refund processed within 5-7 business days after receiving returned item - Original

Query 2: 'How many days do I have to return an item?'

--- Result 1 (score: 0.4372) ---
  Source: s3://composable-agents-kb-359894365202-us-west-2/return-polic

## Step 10: Store KB ID in SSM Parameter Store

Publish the Knowledge Base ID to SSM Parameter Store so the Refund Agent in L3 can look it up at runtime without hardcoding.

In [18]:
SSM_PREFIX = "/anycompany/agentcore"

ssm = boto3.client("ssm", region_name=REGION)

ssm.put_parameter(
    Name=f"{SSM_PREFIX}/kb_id",
    Value=KB_ID,
    Type="String",
    Overwrite=True,
)
print(f"Stored: {SSM_PREFIX}/kb_id = {KB_ID}")

Stored: /anycompany/agentcore/kb_id = BAQBCKQYIC


Print the key resource IDs created in this notebook for reference.

In [19]:
print("Save these for use in downstream agents:")
print(f"  KB_ID              = \"{KB_ID}\"")
print(f"  S3_BUCKET          = \"{BUCKET_NAME}\"")
print(f"  COLLECTION_ARN     = \"{COLLECTION_ARN}\"")
print(f"  KB_ROLE_ARN        = \"{KB_ROLE_ARN}\"")

Save these for use in downstream agents:
  KB_ID              = "BAQBCKQYIC"
  S3_BUCKET          = "composable-agents-kb-359894365202-us-west-2"
  COLLECTION_ARN     = "arn:aws:aoss:us-west-2:359894365202:collection/t5x2c9c2o4p1opriibia"
  KB_ROLE_ARN        = "arn:aws:iam::359894365202:role/composable-agents-kb-role"


## Summary

| Resource | Value |
|----------|-------|
| S3 Data Bucket | `composable-agents-kb-{ACCOUNT_ID}-{REGION}` |
| OpenSearch Collection | `composable-agents-collection` |
| Vector Index | `bedrock-knowledge-base-default-index` |
| Embedding Model | `amazon.titan-embed-text-v2:0` (FLOAT32, 1024 dimensions) |
| Knowledge Base | Use `KB_ID` in downstream agents |

This knowledge base uses Titan Text Embeddings V2 to embed the return-policy documents, enabling semantic text search that the L3 Refund Agent queries at runtime.

---
## Cleanup

Run the following cell to delete all resources created in this notebook and avoid ongoing charges.

**Resources deleted:**
- Bedrock Data Source and Knowledge Base
- OpenSearch Serverless collection and policies
- S3 bucket (data)
- IAM policy and role
- SSM parameter

Uncomment and run this cell to delete all resources created in this notebook. Only run after completing all other notebooks.

In [20]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.

# import boto3, json, os, time
#
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
# SSM_PREFIX = "/anycompany/agentcore"
#
# BUCKET_NAME = f"composable-agents-kb-{ACCOUNT_ID}-{REGION}"
# COLLECTION_NAME = "composable-agents-collection"
# KB_ROLE_NAME = "composable-agents-kb-role"
# POLICY_NAME = "composable-agents-kb-policy"
# POLICY_ARN = f"arn:aws:iam::{ACCOUNT_ID}:policy/{POLICY_NAME}"
# KB_NAME = "composable-agents-kb"
#
# bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)
# aoss = boto3.client("opensearchserverless", region_name=REGION)
# s3 = boto3.client("s3", region_name=REGION)
# iam = boto3.client("iam")
# ssm = boto3.client("ssm", region_name=REGION)
#
# # Order matters: children before parents. IAM role and OSS collection must
# # outlive the KB so its async delete can finish — they get deleted last.
#
# # ── Phase 1: Find KB by name (idempotent on re-runs) ──────────────────────
# kb_id = None
# try:
#     for kb in bedrock_agent.list_knowledge_bases().get("knowledgeBaseSummaries", []):
#         if kb["name"] == KB_NAME:
#             kb_id = kb["knowledgeBaseId"]
#             break
# except Exception as e:
#     print(f"List KBs: {e}")
#
# # ── Phase 2: Drain data sources (force RETAIN so vector-store cleanup is skipped) ──
# if kb_id:
#     try:
#         ds_list = bedrock_agent.list_data_sources(knowledgeBaseId=kb_id).get("dataSourceSummaries", [])
#         for ds in ds_list:
#             ds_id = ds["dataSourceId"]
#             try:
#                 detail = bedrock_agent.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]
#                 bedrock_agent.update_data_source(
#                     knowledgeBaseId=kb_id,
#                     dataSourceId=ds_id,
#                     name=detail["name"],
#                     dataDeletionPolicy="RETAIN",
#                     dataSourceConfiguration=detail["dataSourceConfiguration"],
#                 )
#             except Exception:
#                 pass  # best-effort; safe to skip if already RETAIN
#             bedrock_agent.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
#             print(f"Initiated delete of data source: {ds_id}")
#         deadline = time.time() + 300  # 5 min
#         while time.time() < deadline:
#             remaining = bedrock_agent.list_data_sources(knowledgeBaseId=kb_id).get("dataSourceSummaries", [])
#             if not remaining:
#                 print("All data sources deleted.")
#                 break
#             time.sleep(10)
#         else:
#             print("Timed out waiting for data sources to delete; continuing anyway.")
#     except Exception as e:
#         print(f"Data source drain: {e}")
#
# # ── Phase 3: Delete KB and poll until fully gone ──────────────────────────
# if kb_id:
#     try:
#         bedrock_agent.delete_knowledge_base(knowledgeBaseId=kb_id)
#         print(f"Initiated delete of Knowledge Base: {kb_id}")
#         deadline = time.time() + 600  # 10 min
#         while time.time() < deadline:
#             try:
#                 status = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"]
#                 print(f"  KB status: {status}")
#             except bedrock_agent.exceptions.ResourceNotFoundException:
#                 print(f"Knowledge Base {kb_id} fully deleted.")
#                 break
#             time.sleep(15)
#         else:
#             print("Timed out waiting for KB to delete; continuing — manual cleanup may be needed.")
#     except Exception as e:
#         print(f"KB delete: {e}")
#
# # ── Phase 4: OSS collection + policies (safe now that KB is gone) ─────────
# try:
#     collections = aoss.batch_get_collection(names=[COLLECTION_NAME]).get("collectionDetails", [])
#     if collections:
#         aoss.delete_collection(id=collections[0]["id"])
#         print(f"Deleted OpenSearch collection: {COLLECTION_NAME}")
# except Exception as e:
#     print(f"Collection cleanup: {e}")
#
# try:
#     aoss.delete_access_policy(name="comp-agents-kb-access", type="data")
#     print("Deleted data access policy")
# except Exception as e:
#     print(f"Data access policy cleanup: {e}")
#
# for sec_name, sec_type in [(f"{COLLECTION_NAME}-net", "network"), (f"{COLLECTION_NAME}-enc", "encryption")]:
#     try:
#         aoss.delete_security_policy(name=sec_name, type=sec_type)
#         print(f"Deleted {sec_type} policy: {sec_name}")
#     except Exception as e:
#         print(f"{sec_type} policy cleanup ({sec_name}): {e}")
#
# # ── Phase 5: S3 buckets (empty then delete) ───────────────────────────────
# for bucket in [BUCKET_NAME]:
#     try:
#         paginator = s3.get_paginator("list_objects_v2")
#         for page in paginator.paginate(Bucket=bucket):
#             if "Contents" in page:
#                 objects = [{"Key": obj["Key"]} for obj in page["Contents"]]
#                 s3.delete_objects(Bucket=bucket, Delete={"Objects": objects})
#         s3.delete_bucket(Bucket=bucket)
#         print(f"Deleted S3 bucket: {bucket}")
#     except Exception as e:
#         print(f"S3 cleanup ({bucket}): {e}")
#
# # ── Phase 6: IAM (LAST — KB needed this role to delete itself in Phase 3) ─
# try:
#     iam.detach_role_policy(RoleName=KB_ROLE_NAME, PolicyArn=POLICY_ARN)
#     print("Detached policy from role")
# except Exception as e:
#     print(f"Detach policy: {e}")
#
# try:
#     versions = iam.list_policy_versions(PolicyArn=POLICY_ARN)["Versions"]
#     for v in versions:
#         if not v["IsDefaultVersion"]:
#             iam.delete_policy_version(PolicyArn=POLICY_ARN, VersionId=v["VersionId"])
#     iam.delete_policy(PolicyArn=POLICY_ARN)
#     print(f"Deleted IAM policy: {POLICY_NAME}")
# except Exception as e:
#     print(f"Policy cleanup: {e}")
#
# try:
#     iam.delete_role(RoleName=KB_ROLE_NAME)
#     print(f"Deleted IAM role: {KB_ROLE_NAME}")
# except Exception as e:
#     print(f"Role cleanup: {e}")
#
# # ── Phase 7: SSM parameter ────────────────────────────────────────────────
# try:
#     ssm.delete_parameter(Name=f"{SSM_PREFIX}/kb_id")
#     print(f"Deleted SSM parameter: {SSM_PREFIX}/kb_id")
# except Exception as e:
#     print(f"SSM cleanup: {e}")
#
# print("\n✅ All Notebook 1 resources cleaned up.")